In [1]:
import os
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report


In [24]:
DATA_DIR = Path(r"C:\Users\User\OneDrive - Ministere de l'Enseignement Superieur et de la Recherche Scientifique\Bureau\NLP\data\raw\aclImdb")

In [25]:
def load_split(split_dir):
    texts = []
    labels = []

    pos_dir = split_dir / "pos"
    neg_dir = split_dir / "neg"

    for fname in sorted(pos_dir.glob("*.txt")):
        with open(fname, "r", encoding="utf-8", errors="ignore") as f:
            texts.append(f.read())
            labels.append(1)

    for fname in sorted(neg_dir.glob("*.txt")):
        with open(fname, "r", encoding="utf-8", errors="ignore") as f:
            texts.append(f.read())
            labels.append(0)

    return texts, labels

In [26]:


def main():
    train_dir = DATA_DIR / "train"
    test_dir = DATA_DIR / "test"

    print("Loading data...")
    X_train, y_train = load_split(train_dir)
    X_test, y_test = load_split(test_dir)

    print(f"Loaded {len(X_train)} train docs, {len(X_test)} test docs.")

    # --- TF–IDF vectorization ---
    # This is your "just tf-idf" representation; no extra BoW step needed.
    vectorizer = TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 1),   # start with unigrams
        min_df=5,             # ignore very rare words; adjust if needed
        max_df=0.95,          # drop extremely frequent terms
        stop_words=None       # you can experiment with 'english'
    )

    print("Fitting TF–IDF on training data...")
    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_test_tfidf = vectorizer.transform(X_test)

    print(f"Train TF–IDF shape: {X_train_tfidf.shape}")
    print(f"Test  TF–IDF shape: {X_test_tfidf.shape}")

    # --- Classifier ---
    clf = LogisticRegression(
        max_iter=1000,
        solver="liblinear"  # good for sparse, binary problems
    )

    print("Training classifier...")
    clf.fit(X_train_tfidf, y_train)

    print("Evaluating...")
    y_pred = clf.predict(X_test_tfidf)
    acc = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {acc:.4f}")
    print("\nDetailed report:")
    print(classification_report(y_test, y_pred, digits=4))


In [27]:


if __name__ == "__main__":
    main()

Loading data...
Loaded 25000 train docs, 25000 test docs.
Fitting TF–IDF on training data...
Train TF–IDF shape: (25000, 27270)
Test  TF–IDF shape: (25000, 27270)
Training classifier...
Evaluating...
Accuracy: 0.8849

Detailed report:
              precision    recall  f1-score   support

           0     0.8838    0.8862    0.8850     12500
           1     0.8859    0.8835    0.8847     12500

    accuracy                         0.8849     25000
   macro avg     0.8849    0.8849    0.8849     25000
weighted avg     0.8849    0.8849    0.8849     25000

